# ONH Large- and Small-Vessel Separation Experiments

This notebook compares ONH large/small vessel separation strategies. It does not change the GUI, Retina code, or the production ONH pipeline.

The pipeline is fixed:

1. Selected ONH binarisation method.
2. Overall binary vessel mask within the fixed ONH annulus.
3. Distance-transform diameter map.
4. Compare three diameter-threshold strategies for separating large and small vessels.

The goal is to select a practical default threshold for ONH large-vessel segmentation while preserving the capillary network in the small-vessel mask.

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.morphology import remove_small_objects, skeletonize

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "gui" / "build" / "modules" / "onh_binarise_image.py").exists():
            if (candidate / "algorithms test images").exists():
                return candidate
    raise FileNotFoundError("Could not find the OCTAVA project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TEST_DIR = PROJECT_ROOT / "algorithms test images"
GUI_BUILD_DIR = PROJECT_ROOT / "gui" / "build"
MODULE_DIR = GUI_BUILD_DIR / "modules"

for item in (str(GUI_BUILD_DIR), str(MODULE_DIR)):
    if item not in sys.path:
        sys.path.insert(0, item)

from modules.onh_binarise_image import binarise_onh_image

METADATA_PATH = TEST_DIR / "test images information.xlsx"
OUTPUT_DIR = TEST_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)

## Load Test Images and Annular ROI

The annular ROI is fixed at 2-4 mm diameter within a 4.5 mm x 4.5 mm ONH scan.

In [ ]:
IMAGE_SIZE_PX = 1500
SCAN_SIZE_MM = 4.5
INNER_DIAMETER_MM = 2.0
OUTER_DIAMETER_MM = 4.0

metadata = pd.read_excel(METADATA_PATH)
metadata

In [ ]:
def read_grayscale_image(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float64)


def resize_to_scan(image: np.ndarray, image_size_px: int = IMAGE_SIZE_PX) -> np.ndarray:
    interpolation = cv2.INTER_AREA if max(image.shape) > image_size_px else cv2.INTER_CUBIC
    return cv2.resize(image, (image_size_px, image_size_px), interpolation=interpolation).astype(np.float64)


def annulus_mask(image_shape, center_x, center_y):
    h, w = image_shape
    px_per_mm = w / SCAN_SIZE_MM
    inner_radius_px = (INNER_DIAMETER_MM / 2.0) * px_per_mm
    outer_radius_px = (OUTER_DIAMETER_MM / 2.0) * px_per_mm
    yy, xx = np.ogrid[:h, :w]
    distance = np.sqrt((xx - center_x) ** 2 + (yy - center_y) ** 2)
    return (distance >= inner_radius_px) & (distance <= outer_radius_px)


def load_cases(metadata: pd.DataFrame) -> list[dict]:
    cases = []
    for _, row in metadata.iterrows():
        image_path = TEST_DIR / str(row["filename"])
        raw = read_grayscale_image(image_path)
        image = resize_to_scan(raw)
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        device = str(row["device"]).strip()
        quality = str(row["quality"]).strip()
        cases.append({
            "name": f"{device} {quality}",
            "image": image,
            "roi_mask": annulus_mask(image.shape, center_x, center_y),
            "center_x": center_x,
            "center_y": center_y,
        })
    return cases


cases = load_cases(metadata)
[(case["name"], case["image"].shape) for case in cases]

## Generate Overall Binary Masks

All separation strategies use the same selected ONH binarisation result.

In [ ]:
class NotebookVar:
    def __init__(self, value):
        self.value = value
    def get(self):
        return self.value


class NotebookCanvas:
    status_id = 1
    frangi_bool = NotebookVar(1)
    frangi_min = NotebookVar(1)
    frangi_max = NotebookVar(4)
    med_filt_bool = NotebookVar(0)
    kernel_size = NotebookVar(3)

    def itemconfigure(self, *args, **kwargs):
        pass

    def update_idletasks(self):
        pass


def generate_binary(case: dict) -> np.ndarray:
    canvas = NotebookCanvas()
    canvas.onh_mask_img = case["image"].astype(np.float64)
    canvas.onh_mask_arr = case["roi_mask"].astype(bool)
    binarise_onh_image(canvas, display=False)
    return canvas.binary_image.astype(np.uint8)


binary_results = {case["name"]: generate_binary(case) for case in cases}
print("Generated selected ONH binary masks.")

## Separation Strategies

The labels below are qualitative and report-friendly.

- **Restrictive large-vessel mask (50 um)**: only the most obvious large vessels are classified as large.
- **Balanced large-vessel mask (40 um)**: current conservative default.
- **Sensitive large-vessel mask (30 um)**: includes more medium-sized vessels in the large-vessel class.

In [ ]:
SEPARATION_STRATEGIES = {
    "Restrictive large-vessel mask (50 um)": 50.0,
    "Balanced large-vessel mask (40 um)": 40.0,
    "Sensitive large-vessel mask (30 um)": 30.0,
}

SEPARATION_STRATEGIES

In [ ]:
def separate_vessels_by_threshold(binary: np.ndarray, roi_mask: np.ndarray, threshold_um: float) -> dict:
    overall_mask = (binary > 0) & roi_mask.astype(bool)
    height, width = overall_mask.shape
    pixel_size_mm = SCAN_SIZE_MM / float(width)

    radius_map_px = distance_transform_edt(overall_mask)
    diameter_map_um = radius_map_px * 2.0 * pixel_size_mm * 1000.0

    large_seed = overall_mask & (diameter_map_um >= threshold_um)

    threshold_mm = threshold_um / 1000.0
    large_radius_px = max(1, int(round((threshold_mm / pixel_size_mm) / 2.0)))

    large_mask = binary_dilation(large_seed, iterations=large_radius_px) & overall_mask
    min_large_area = max(20, int(np.pi * (large_radius_px ** 2) * 0.5))
    large_mask = remove_small_objects(large_mask.astype(bool), min_size=min_large_area, connectivity=2)
    large_mask = large_mask & overall_mask & roi_mask

    small_mask = overall_mask & (~large_mask) & roi_mask

    return {
        "overall_mask": overall_mask,
        "large_seed": large_seed,
        "large_mask": large_mask,
        "small_mask": small_mask,
        "diameter_map_um": diameter_map_um,
        "large_radius_px": large_radius_px,
        "min_large_area": min_large_area,
    }


separation_results = {}
for strategy_name, threshold_um in SEPARATION_STRATEGIES.items():
    separation_results[strategy_name] = {}
    for case in cases:
        separation_results[strategy_name][case["name"]] = separate_vessels_by_threshold(
            binary_results[case["name"]],
            case["roi_mask"],
            threshold_um,
        )

print("Completed large/small vessel separation strategies.")

## Quality Control Table

This table checks whether large and small masks fully partition the overall binary mask and how much vessel area is assigned to each class.

In [ ]:
def separation_qc_table(cases: list[dict], separation_results: dict) -> pd.DataFrame:
    rows = []
    for strategy_name, by_case in separation_results.items():
        threshold_um = SEPARATION_STRATEGIES[strategy_name]
        for case in cases:
            result = by_case[case["name"]]
            overall = result["overall_mask"]
            large = result["large_mask"]
            small = result["small_mask"]
            overlap = large & small
            reconstructed = large | small
            missing = overall & (~reconstructed)
            extra = reconstructed & (~overall)
            overall_area = int(np.count_nonzero(overall))
            large_area = int(np.count_nonzero(large))
            small_area = int(np.count_nonzero(small))
            rows.append({
                "case": case["name"],
                "strategy": strategy_name,
                "threshold_um": threshold_um,
                "overall_vessel_pixels": overall_area,
                "large_vessel_pixels": large_area,
                "small_vessel_pixels": small_area,
                "large_fraction_of_vessel_area_pct": 100 * large_area / max(overall_area, 1),
                "small_fraction_of_vessel_area_pct": 100 * small_area / max(overall_area, 1),
                "overlap_pixels": int(np.count_nonzero(overlap)),
                "missing_pixels": int(np.count_nonzero(missing)),
                "extra_pixels": int(np.count_nonzero(extra)),
                "large_radius_px": result["large_radius_px"],
            })
    return pd.DataFrame(rows)


separation_qc = separation_qc_table(cases, separation_results)
separation_qc

## Report Figure 1: Large/Small Vessel Overlay

This is the main visual comparison. Red pixels are large vessels and cyan pixels are small vessels.

In [ ]:
REPORT_FIGURE_1 = OUTPUT_DIR / "onh_large_small_vessel_separation_overlay.png"


def separation_overlay(result: dict) -> np.ndarray:
    large = result["large_mask"]
    small = result["small_mask"]
    rgb = np.zeros((*large.shape, 3), dtype=np.uint8)
    rgb[small] = np.array([60, 220, 240], dtype=np.uint8)   # cyan
    rgb[large] = np.array([255, 80, 70], dtype=np.uint8)    # red
    return rgb


def plot_separation_overlay_grid(cases: list[dict], separation_results: dict, save_path: Path | None = None) -> None:
    strategy_names = list(separation_results.keys())
    fig, axes = plt.subplots(
        len(cases),
        len(strategy_names),
        figsize=(4.2 * len(strategy_names), 3.9 * len(cases)),
        squeeze=False,
    )

    for r, case in enumerate(cases):
        for c, strategy_name in enumerate(strategy_names):
            ax = axes[r, c]
            result = separation_results[strategy_name][case["name"]]
            ax.imshow(separation_overlay(result))
            if r == 0:
                ax.set_title(strategy_name)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle("ONH large- and small-vessel separation strategy comparison", y=0.995)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()


plot_separation_overlay_grid(cases, separation_results, save_path=REPORT_FIGURE_1)

## Report Figure 2: Area Assigned to Large Vessels

This figure summarises how strongly each threshold changes the large-vessel class.

In [ ]:
REPORT_FIGURE_2 = OUTPUT_DIR / "onh_large_vessel_area_fraction_by_threshold.png"

pivot = separation_qc.pivot(index="case", columns="strategy", values="large_fraction_of_vessel_area_pct")
pivot = pivot[list(SEPARATION_STRATEGIES.keys())]
colors = ["#c75b5b", "#2f80ed", "#69b36d"]
ax = pivot.plot(kind="bar", figsize=(10, 4.4), width=0.78, color=colors)
ax.set_ylabel("Large-vessel area (% of total vessel area)")
ax.set_title("Effect of diameter threshold on ONH large-vessel area")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Separation strategy", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(REPORT_FIGURE_2, bbox_inches="tight")
plt.show()
print("Saved:", REPORT_FIGURE_2)

## Skeleton Preview for Large and Small Vessel Masks

This optional view checks whether the separated masks produce meaningful skeletons for downstream metrics. It is mainly for method tuning, not necessarily for the report.

In [ ]:
def quick_skeleton(mask: np.ndarray) -> np.ndarray:
    cleaned = remove_small_objects(mask.astype(bool), min_size=20, connectivity=2)
    return skeletonize(cleaned)


def plot_selected_strategy_skeleton_preview(strategy_name: str = "Balanced large-vessel mask (40 um)") -> None:
    fig, axes = plt.subplots(len(cases), 2, figsize=(7.8, 3.7 * len(cases)), squeeze=False)
    for r, case in enumerate(cases):
        result = separation_results[strategy_name][case["name"]]
        large_skel = binary_dilation(quick_skeleton(result["large_mask"]), iterations=1)
        small_skel = binary_dilation(quick_skeleton(result["small_mask"]), iterations=1)
        for c, (title, skel) in enumerate([("Large-vessel skeleton", large_skel), ("Small-vessel skeleton", small_skel)]):
            ax = axes[r, c]
            ax.imshow(skel, cmap="gray")
            if r == 0:
                ax.set_title(title)
            if c == 0:
                ax.set_ylabel(case["name"], rotation=90, labelpad=8)
            ax.axis("off")
    fig.suptitle(f"Skeleton preview for {strategy_name}", y=0.995)
    plt.tight_layout()


plot_selected_strategy_skeleton_preview()

## Decision Guidance

Use the overlay and area-fraction figure to decide the default ONH large-vessel threshold.

A suitable threshold should:

1. Keep the main radial vessels in the large-vessel mask.
2. Avoid assigning too much of the dense capillary network to the large-vessel class.
3. Leave the small-vessel mask with a plausible capillary network.
4. Fully partition the overall binary mask, with no overlap or missing vessel pixels.

The current production default is **Balanced large-vessel mask (40 um)**. If the overlay shows that this threshold best balances large-vessel continuity and capillary preservation, it can remain as the ONH default.